# Country Maps
Generates a styled JPG map for each SIDS country showing the country outline and tile grid.

In [19]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

%matplotlib inline

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.ticker import FuncFormatter
import contextily as ctx
import pyproj


# I already ran this once so no need to rerun.
# get_gadm(countries=ALL_COUNTRIES, overwrite=True) # Update to all countries.

from dep_tools.grids import (
    COUNTRIES_AND_CODES as DEP_COUNTRIES_AND_CODES,
)
from ldn.utils import NON_DEP_COUNTRIES

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
COUNTRIES_PATH = Path("../../ldn/gadm_sids.gpkg")
GRID_PATH = Path("../../ldn/sids_all_tiles.geojson")
OUTPUT_DIR = Path("../maps")

for subdir in ("countries/Pacific", "countries/Non-Pacific", "regions"):
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

STANDARD_CRS = "EPSG:3857"
PACIFIC_CRS = "EPSG:3832"

NAME_COL = "COUNTRY"
countries_src = gpd.read_file(COUNTRIES_PATH)
grid_src = gpd.read_file(GRID_PATH)
print(f"Loaded {len(countries_src)} countries, {len(grid_src)} grid tiles")

crs_layers = {
    STANDARD_CRS: (countries_src.to_crs(STANDARD_CRS), grid_src.to_crs(STANDARD_CRS)),
    PACIFIC_CRS: (countries_src.to_crs(PACIFIC_CRS), grid_src.to_crs(PACIFIC_CRS)),
}

DEP_COUNTRY_NAMES = set(DEP_COUNTRIES_AND_CODES.keys())
NON_DEP_COUNTRY_NAMES = set(NON_DEP_COUNTRIES.keys())

REGION_CRS = {
    "Pacific": PACIFIC_CRS,
    "Non-Pacific": STANDARD_CRS,
}
REGION_COLORS = {
    "Pacific": {"boundary": "#FF003C", "grid": "#00CFFF"},
    "Non-Pacific": {"boundary": "#FF6B00", "grid": "#39FF14"},
}
REGION_COUNTRIES = {
    "Pacific": DEP_COUNTRY_NAMES,
    "Non-Pacific": NON_DEP_COUNTRY_NAMES,
}

MARGIN = 1.0
PADDING = 0.7
C = {"grey": "#e1e1e1", "black": "#000000", "white": "#FFFFFF"}
out_dpi = 75

Loaded 60 countries, 817 grid tiles


In [21]:
def get_region_label(country_name: str) -> str:
    """Return the region name for a given country."""
    return "Pacific" if country_name in DEP_COUNTRY_NAMES else "Non-Pacific"


def get_basemap_zoom(title: str) -> int | str:
    """Return basemap zoom level. Most use auto; antimeridian-crossing archipelagos need a manual override."""
    overrides = {"Pacific": 4, "Fiji": 8, "Kiribati": 5, "Tuvalu": 8}
    return overrides.get(title, "auto")


def add_north_arrow(ax: plt.Axes) -> None:
    """Add a north arrow to the upper right of the axes."""
    x, y, size = 0.97, 0.9, 0.055
    ax.annotate(
        "",
        xy=(x, y + size),
        xytext=(x, y),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->", color=C["grey"], lw=2.5),
        zorder=5,
    )
    ax.text(
        x,
        y + size + 0.005,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color=C["white"],
        path_effects=[pe.withStroke(linewidth=2, foreground=C["black"])],
        zorder=5,
    )


def make_map(
    title: str,
    subtitle: str,
    geom: gpd.GeoDataFrame,
    grid: gpd.GeoDataFrame,
    output_path: Path,
    color_dict: dict,
    crs: str,
    mode: str,
) -> None:
    """Render and save a styled map. geom and grid must already be in crs."""
    color_boundary, color_grid = color_dict["boundary"], color_dict["grid"]
    local_tf = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

    if mode == "Country":
        padding = 0.2
    else:
        padding = 0.03

    boundary_label = f"{mode} boundary"

    ref = grid if not grid.empty else geom
    bounds = ref.total_bounds
    min_extent = max(bounds[2] - bounds[0], bounds[3] - bounds[1], 50_000)
    half = (min_extent * (1 + padding)) / 2
    cx, cy = (bounds[0] + bounds[2]) / 2, (bounds[1] + bounds[3]) / 2

    fig, ax = plt.subplots(figsize=(10, 10), dpi=out_dpi)
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)

    zoom = get_basemap_zoom(title)
    print(f"  {title}: zoom={zoom}")
    ctx.add_basemap(
        ax,
        crs=crs,
        source=ctx.providers.Esri.WorldImagery,
        zoom=zoom,
        attribution=False,
    )

    if not grid.empty:
        grid.plot(ax=ax, facecolor="none", edgecolor=color_grid, linewidth=1.8)
    geom.plot(ax=ax, facecolor="none", edgecolor=color_boundary, linewidth=2.5)

    if mode == "Country":
        padding = 0.2
        ax.text(
            0.5,
            1.02,
            f"{title} - {subtitle}",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=18,
            fontweight="bold",
            color=C["black"],
        )
        ax.text(
            0.99,
            1.02,
            "\u25cf",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=18,
            color=color_grid,
            path_effects=[pe.withStroke(linewidth=2, foreground=C["black"])],
        )
    else:
        padding = 0.03
        ax.text(
            0.5,
            1.02,
            f"{title} - {subtitle}",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=18,
            fontweight="bold",
            color=C["black"],
        )
        ax.text(
            0.01,
            1.02,
            "\u25cf",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=18,
            color=color_grid,
            path_effects=[pe.withStroke(linewidth=2, foreground=C["black"])],
        )

    # Workaround to get km instead of Mm at small scale.
    extent_km = (half * 2) / 1000
    scale_km = next(
        v
        for v in [5000, 2000, 1000, 500, 200, 100, 50, 20, 10, 5, 2, 1]
        if v <= extent_km / 4
    )
    ax.add_artist(
        ScaleBar(
            1,
            units="m",
            fixed_value=scale_km,
            fixed_units="km",
            location="lower left",
            pad=PADDING,
            border_pad=MARGIN,
            sep=3,
            color=C["white"],
            box_color=C["black"],
            box_alpha=0.7,
            font_properties={"size": 11},
        )
    )

    add_north_arrow(ax)

    ax.legend(
        handles=[
            Line2D([0], [0], color=color_boundary, linewidth=2.5, label=boundary_label),
            Line2D(
                [0], [0], color=color_grid, linewidth=1.8, label="Analysis grid tiles"
            ),
        ],
        loc="upper left",
        framealpha=0.7,
        fontsize=11,
        labelcolor=C["white"],
        facecolor=C["black"],
        borderpad=PADDING,
        borderaxespad=MARGIN,
        handlelength=1.5,
    )

    def fmt_lon(x: float, _: int) -> str:
        """Format a projected x coordinate as a longitude string."""
        lon, _ = local_tf.transform(x, 0)
        return f"{lon:.2f}E" if lon >= 0 else f"{abs(lon):.2f}W"

    def fmt_lat(y: float, _: int) -> str:
        """Format a projected y coordinate as a latitude string."""
        _, lat = local_tf.transform(0, y)
        return f"{lat:.2f}N" if lat >= 0 else f"{abs(lat):.2f}S"

    ax.xaxis.set_major_formatter(FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_lat))
    ax.tick_params(labelsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_edgecolor(C["black"])
    ax.text(
        0.99,
        0.01,
        "Basemap: Esri World Imagery | Boundaries: GADM",
        transform=ax.transAxes,
        fontsize=7,
        color=C["grey"],
        va="bottom",
        ha="right",
    )

    plt.tight_layout()
    fig.savefig(
        output_path,
        dpi=out_dpi,
        format="jpg",
        bbox_inches="tight",
        pil_kwargs={"quality": 92},
    )
    plt.close(fig)
    print(f"  Saved: {output_path}")


def make_country_map(country_name: str, output_path: Path) -> None:
    """Render and save a map for a single country."""
    region = get_region_label(country_name)
    crs = REGION_CRS[region]
    countries_proj, grid_proj = crs_layers[crs]
    geom = countries_proj[countries_proj[NAME_COL] == country_name]
    tiles = grid_proj[grid_proj.intersects(geom.union_all())]
    make_map(
        country_name,
        region,
        geom,
        tiles,
        output_path,
        REGION_COLORS[region],
        crs,
        mode="Country",
    )


def make_region_map(region_name: str, output_path: Path) -> None:
    """Render and save a map for all countries in a region."""
    crs = REGION_CRS[region_name]
    countries_proj, grid_proj = crs_layers[crs]
    geom = countries_proj[countries_proj[NAME_COL].isin(REGION_COUNTRIES[region_name])]
    tiles = grid_proj[grid_proj.intersects(geom.union_all())]
    make_map(
        region_name,
        f"{len(geom)} countries / territories",
        geom,
        tiles,
        output_path,
        REGION_COLORS[region_name],
        crs,
        mode="Region",
    )

In [22]:
country_names = sorted(countries_src[NAME_COL].unique())
print(f"Generating {len(country_names)} country maps...")
for name in country_names:
    safe = name.replace(" ", "_").replace("/", "-")
    out = OUTPUT_DIR / "countries" / get_region_label(name) / f"{safe}.jpg"
    try:
        make_country_map(name, out)
    except Exception as e:
        print(f"  ERROR {name}: {e}")

print("\nGenerating region maps...")
for region_name in REGION_COUNTRIES:
    try:
        make_region_map(region_name, OUTPUT_DIR / "regions" / f"{region_name}.jpg")
    except Exception as e:
        print(f"  ERROR {region_name}: {e}")

print(f"\nDone: {OUTPUT_DIR}")

Generating 60 country maps...
  American Samoa: zoom=auto
  Saved: ../maps/countries/Pacific/American_Samoa.jpg
  Anguilla: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Anguilla.jpg
  Antigua and Barbuda: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Antigua_and_Barbuda.jpg
  Aruba: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Aruba.jpg
  Bahamas: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Bahamas.jpg
  Barbados: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Barbados.jpg
  Belize: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Belize.jpg
  Bermuda: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Bermuda.jpg
  British Virgin Islands: zoom=auto
  Saved: ../maps/countries/Non-Pacific/British_Virgin_Islands.jpg
  Cabo Verde: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Cabo_Verde.jpg
  Cayman Islands: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Cayman_Islands.jpg
  Comoros: zoom=auto
  Saved: ../maps/countries/Non-Pacific/Comoros.jpg
  Cook Islands: zoom=aut